# Notebook 2: Logit Lens
**Kurs:** Mechanistic Interpretability  
**Modell:** EleutherAI/pythia-410m  
**Ziel:** Vorhersagen nach jeder Schicht visualisieren.

## Hintergrund: Was ist der Logit Lens?

Der **Logit Lens** (nostalgebraist, 2020) ist eine Technik, um zu sehen, was ein Transformer "nach jeder Schicht denkt".

**Kernidee:** Transformer haben einen **Residual Stream** — ein Informationskanal, der durch alle Schichten fließt. Jede Schicht *addiert* ihren Beitrag.

Da der finale Output die letzte Projektion des Residual Streams auf den Vokabular-Raum ist, können wir dieselbe **Unembed-Matrix W_U** auch auf frühere Hidden States anwenden:

```
Logit Lens bei Schicht L: logits_L = hidden_state_L @ W_U.T
```

So sehen wir, wie sich die Vorhersage über die Schichten herausbildet.

**Frage:** In welcher Schicht "weiß" das Modell die richtige Antwort?

## 1. Setup und Modell laden

In [ ]:
%load_ext autoreload
%autoreload 2

# Standard-library imports used only for notebook bootstrapping.
# The analysis logic itself lives in the utility modules imported below.
import sys
import importlib
from pathlib import Path
from IPython.display import Image, display

# The notebook is stored in assignement/notebooks, while reusable Python
# modules are stored in assignement/utils. Adding that folder to sys.path lets
# the notebook import project utilities without duplicating implementation code.
utils_dir = Path.cwd().parent / "utils"
if not (utils_dir / "config.py").exists():
    raise FileNotFoundError(f"Could not find the assignment utils directory: {utils_dir}")

if str(utils_dir) not in sys.path:
    sys.path.insert(0, str(utils_dir))

# Import the small set of modules needed for this notebook. The autoreload
# extension plus explicit reloads make notebook reruns pick up utility edits.
import config
import model_utils
import plotting_utils
import logit_lens_utils

for module in [
    config,
    model_utils,
    plotting_utils,
    logit_lens_utils,
]:
    importlib.reload(module)

# Keep the notebook readable by importing only high-level orchestration
# functions. Heavy computation and plotting stay in the utility files.
from model_utils import load_model_and_tokenizer, setup_environment
from logit_lens_utils import (
    analyze_target_token_probability,
    logit_lens_all_layers,
    print_layer_top1_table,
    print_target_token_findings,
    prepare_hu_liu_sentiment_state,
    print_top_tokens_for_final_layer,
    run_prompt_pair_forward_passes,
    save_logit_lens_heatmaps,
    select_hu_liu_target_records,
    unembedding_matrix,
)

# Select the available device and load the same model/tokenizer pair used in
# the rest of the assignment.
device = setup_environment()
model, tokenizer = load_model_and_tokenizer(device)

# Display helper for PNG files produced by plotting utilities. Plot functions
# save images to config.OUTPUT_PNG_DIR; the notebook only displays them.
def display_image(png_name):
    image_path = config.OUTPUT_PNG_DIR / png_name
    if not image_path.exists():
        raise FileNotFoundError(f"PNG image not found in output directory: {image_path}")
    display(Image(filename=str(image_path)))


## 2. Forward Pass mit Hidden States

In [ ]:
# Define the two sentiment prompts once.
# Every later analysis cell uses this same pair through the prompt_pair object.
positive_prompt = "The movie was wonderful and I felt very"
negative_prompt = "The movie was terrible and I felt very"

# Wrap both strings in the same prompt-pair structure used by the utility
# functions. Do not redefine prompt_pair later in the notebook; changing the
# two variables above is enough to rerun the full Logit Lens comparison.
prompt_pair = {
    "id": "user_defined_positive_negative_pair",
    "positive": positive_prompt,
    "negative": negative_prompt,
}

# Run both prompts through the model with output_hidden_states=True.
# The utility returns one hidden-state tensor for the embedding output plus one
# tensor after each transformer layer, separately for the positive and negative
# prompt.
positive_hidden_states, negative_hidden_states = run_prompt_pair_forward_passes(
    model,
    tokenizer,
    device,
    prompt_pair,
)


### Ergebnisinterpretation

In diesem Abschnitt werden die beiden Sentiment-Prompts bewusst direkt im Notebook definiert. Dadurch ist sofort sichtbar, welche positive und negative Formulierung miteinander verglichen wird, und die Beispiele können ohne Änderung an den Utility-Dateien angepasst werden.

Die ausgegebenen Hidden-State-Formen zeigen, dass das Modell für beide Prompts pro Token einen Vektor im Modellraum erzeugt. Die Form folgt immer dem Schema `(Batch-Größe, Token-Anzahl, Hidden Size)`. Hidden State 0 ist dabei die Embedding-Repräsentation vor den Transformer-Schichten; die folgenden Hidden States zeigen die Repräsentationen nach den einzelnen Schichten.

Für den Logit-Lens-Vergleich ist besonders der letzte Token relevant, weil dort die nächste Vorhersage des Modells entsteht. Die späteren Kapitel projizieren diese Zwischendarstellungen zurück in den Vokabularraum und prüfen, ab welchen Schichten sich positive bzw. negative Fortsetzungen im Modell abzeichnen.


## 3. Logit Lens für den finalen Layer


In [ ]:
# The unembedding matrix projects hidden states back into vocabulary space.
# Logit Lens uses this same output matrix at intermediate layers to inspect
# what token distribution each layer would predict if it had to answer early.
unembedding = unembedding_matrix(model)

# Compute the top-5 next-token predictions for every hidden state of the same
# positive_prompt and negative_prompt defined in chapter 2.
positive_layer_results = logit_lens_all_layers(
    positive_hidden_states,
    model,
    tokenizer,
    top_k=5,
)
negative_layer_results = logit_lens_all_layers(
    negative_hidden_states,
    model,
    tokenizer,
    top_k=5,
)

# Prepare the filtered Hu & Liu one-token lexicon once. The same state is
# reused for top-token highlighting and target-token selection below.
hu_liu_sentiment_state = prepare_hu_liu_sentiment_state(tokenizer)

# Print final-layer predictions for both prompts as a sanity check before the
# layer-by-layer comparison. Tokens found in Hu & Liu are highlighted.
print_top_tokens_for_final_layer(
    positive_prompt,
    positive_layer_results,
    hu_liu_sentiment_state,
)
print_top_tokens_for_final_layer(
    negative_prompt,
    negative_layer_results,
    hu_liu_sentiment_state,
)


### Ergebnisinterpretation

Der finale Layer zeigt, welche nächsten Tokens das Modell nach dem positiven und dem negativen Prompt tatsächlich bevorzugt. Durch die Markierung der Hu-&-Liu-Treffer wird zusätzlich sichtbar, ob unter den wahrscheinlichsten Tokens explizite Sentiment-Wörter vorkommen.

Wenn ein Top-K-Token im Hu-&-Liu-Lexikon gefunden wird, kann es direkt als positiver oder negativer Hinweis gelesen werden. Falls keine Treffer auftauchen, bedeutet das nicht automatisch, dass das Modell kein Sentiment repräsentiert; es heißt nur, dass die wahrscheinlichsten nächsten Tokens nicht in dieser lexikonbasierten Wortliste enthalten sind.


## 4. Logit Lens über alle Schichten


In [ ]:
# Compare the top-1 token predicted by Logit Lens at every layer for the
# same positive_prompt and negative_prompt pair from chapter 2.
print_layer_top1_table(positive_layer_results, negative_layer_results)

# Select positive and negative target tokens from the already prepared
# Hu & Liu one-token sentiment state.
hu_liu_target_records = select_hu_liu_target_records(
    sentiment_state=hu_liu_sentiment_state,
)

# Inspect the positive Hu & Liu target token in the layer table and probability
# curve. The negative target token is loaded and printed as the paired contrast.
target_token = hu_liu_target_records["positive"]["token"]
print_target_token_findings(positive_layer_results, target_token)


### Ergebnisinterpretation

Die Schicht-für-Schicht-Tabelle zeigt, wie sich die wahrscheinlichste Fortsetzung im Verlauf des Modells verändert. Frühe Schichten liefern oft noch unspezifische oder instabile Tokens, weil dort vor allem lokale und oberflächennahe Informationen verarbeitet werden.

In späteren Schichten sollten die Top-1-Vorhersagen stärker durch den Semantik- und Sentiment-Kontext der Prompts geprägt sein. Der Vergleich zwischen positivem und negativem Prompt macht sichtbar, ob und wann sich die interne Repräsentation in unterschiedliche sentimentbezogene Fortsetzungen aufspaltet.


## 5. Ziel-Token-Wahrscheinlichkeit pro Schicht


In [ ]:
# Measure the Logit Lens probability of the selected target token at every
# layer for both prompts, then save a line plot to the output PNG folder.
positive_target_probs, negative_target_probs = analyze_target_token_probability(
    positive_hidden_states,
    negative_hidden_states,
    model,
    tokenizer,
    prompt_pair,
    target_token,
)

# Display the PNG saved by the plotting utility. Keeping display separate from
# plotting makes reruns deterministic and avoids rewriting unchanged images.
display_image(config.LOGIT_LENS_TARGET_PROBABILITY_PATH.name)


### Ergebnisinterpretation

Die Ziel-Token-Wahrscheinlichkeit zeigt nicht nur, welches Token am Ende wahrscheinlich ist, sondern wie früh im Modell dieses Token bereits als plausible Fortsetzung angelegt wird. Ein ansteigender Verlauf im positiven Prompt deutet darauf hin, dass die spätere Modellverarbeitung die positive Sentiment-Richtung zunehmend verstärkt.

Der negative Prompt dient als Kontrast. Bleibt die Wahrscheinlichkeit des positiven Ziel-Tokens dort niedriger, spricht das dafür, dass das Modell nicht nur allgemeine Satzstruktur fortsetzt, sondern die Sentiment-Richtung des Prompts in den Zwischendarstellungen berücksichtigt.


## 6. Heatmaps: Top-5 Token je Schicht


In [ ]:
# Save top-5 token heatmaps for the positive and negative prompts. Each row is
# one layer and each column is the token rank within that layer's top-k list.
save_logit_lens_heatmaps(positive_layer_results, negative_layer_results)

# Display the generated heatmaps from the shared output PNG directory.
display_image(config.LOGIT_LENS_POSITIVE_HEATMAP_PATH.name)
display_image(config.LOGIT_LENS_NEGATIVE_HEATMAP_PATH.name)


### Ergebnisinterpretation

Die Heatmaps verdichten die Logit-Lens-Ergebnisse, indem sie pro Schicht die Top-5-Tokens und deren Wahrscheinlichkeiten zeigen. Dadurch lässt sich erkennen, ob einzelne Tokens nur punktuell auftreten oder über mehrere Schichten hinweg stabiler werden.

Besonders interessant ist der Vergleich zwischen positivem und negativem Prompt: Wenn sich die Tokenlisten in den späteren Schichten deutlich unterscheiden, deutet das auf eine sentimentabhängige Entwicklung im Residual Stream hin. Die Heatmap ergänzt damit die Tabellenansicht, weil sie neben dem Top-1-Token auch alternative plausible Fortsetzungen sichtbar macht.


## 7. Reflexionsfragen

1. In welcher Schicht taucht das Ziel-Token erstmals in den Top-5 auf?
2. Was passiert bei den frühen Schichten?
3. Warum bleibt die Ziel-Token-Wahrscheinlichkeit für den negative Prompt niedriger?
4. Was zeigt der Logit Lens über den Residual Stream?
